# Gaussian Shading 官方原始环境复现与 governed import 入口

该 Notebook 服务第二条链路: 官方原始环境复现 + governed import。它不把 legacy Stable Diffusion 结果混入 SD3.5 主表, 只生成补充表所需的官方参考记录、环境报告、命令日志、schema、validation report 和 Google Drive 压缩包。

正式逻辑位于 `paper_workflow/colab_utils/gaussian_shading_official_reference.py`, Notebook 只负责 Colab 冷启动、参数注入、远程执行和打包。


## 运行边界

1. 默认样本数由 `SLM_WM_PAPER_RUN_SAMPLE_COUNT` 决定; `pilot_paper` 对应 600 个 prompt, `full_paper` 对应 6000 个 prompt, 用于 fixed-FPR=0.01 共同协议下的官方 legacy reference 复现。
2. Notebook 会在 helper 中按 `external_baseline/source_registry.json` 自动补齐 Gaussian Shading 官方源码快照; 源码快照仍按外部缓存治理, 不进入 git 提交。
3. 该入口默认把 `SLM_WM_GAUSSIAN_SHADING_OFFICIAL_MODEL_ID` 设为 `Manojb/stable-diffusion-2-1-base`, 该路径是 `stabilityai/stable-diffusion-2-1-base` 不可直接访问时的公开镜像回退。
4. 该入口默认开启 `SLM_WM_GAUSSIAN_SHADING_PATCH_MODEL_REPOSITORY_LAYOUT=1`, 会移除官方源码中对 `revision='fp16'` 分支的硬编码, 因为公开镜像把 fp16 相关文件放在 `main` 而不是 `fp16` 分支。
5. 该入口默认开启 `SLM_WM_GAUSSIAN_SHADING_PREPARE_LOCAL_MODEL_REPOSITORY=1`, 会把公开镜像下载到 `/content/gaussian_shading_model_repository/stable_diffusion_2_1_base`, 并把本地 `model_index.json` 中的 `CLIPImageProcessor` 补丁为 legacy diffusers 可读取的 `CLIPFeatureExtractor`。
6. 该入口默认开启 `SLM_WM_GAUSSIAN_SHADING_PREPARE_LEGACY_ENV=1`, 会先尝试 Python 3.8 + 官方 `requirements.txt` 严格环境; 若官方 requirements 在 Colab 中出现依赖冲突, 再切换到受治理的兼容 fallback 环境。
7. 若 Colab GPU、legacy wheel 或官方依赖组合不可用, helper 会把失败原因写入 `outputs/gaussian_shading_official_reference/` 并继续打包诊断。
8. 若官方原始环境在外部机器完成运行, 可设置 `SLM_WM_GAUSSIAN_SHADING_OFFICIAL_RUN_COMMAND=0`, 并通过 `SLM_WM_GAUSSIAN_SHADING_OFFICIAL_SUMMARY_IMPORT_PATH` 或 `SLM_WM_GAUSSIAN_SHADING_OFFICIAL_LOG_IMPORT_PATH` 导入受治理结果。


In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
SLM_WM_PAPER_RUN_NAME = "pilot_paper"
paper_run_name = SLM_WM_PAPER_RUN_NAME.strip() or "pilot_paper"
os.environ['SLM_WM_PAPER_RUN_NAME'] = paper_run_name
prompt_file_by_run = {
    'pilot_paper': 'configs/paper_main_pilot_paper_prompts.txt',
    'full_paper': 'configs/paper_main_full_paper_prompts.txt',
}
drive_result_root = f'/content/drive/MyDrive/SLM/{paper_run_name}_results'
os.environ['SLM_WM_DRIVE_RESULT_ROOT'] = drive_result_root
paper_run_sample_count = os.environ.get('SLM_WM_PAPER_RUN_SAMPLE_COUNT', 'all')
os.environ['SLM_WM_PAPER_RUN_SAMPLE_COUNT'] = paper_run_sample_count
prompt_count_by_run = {'pilot_paper': 600, 'full_paper': 6000}
def resolve_paper_run_count(value):
    normalized = str(value).strip().lower()
    if normalized in {'', 'all', 'none', 'unlimited'}:
        return int(prompt_count_by_run.get(paper_run_name, prompt_count_by_run['pilot_paper']))
    resolved = int(normalized)
    return int(prompt_count_by_run.get(paper_run_name, prompt_count_by_run['pilot_paper'])) if resolved <= 0 else resolved
paper_run_expected_sample_count = resolve_paper_run_count(paper_run_sample_count)
paper_run_minimum_clean_negative_count = os.environ.get('SLM_WM_PAPER_RUN_MINIMUM_CLEAN_NEGATIVE_COUNT', '100')
os.environ['SLM_WM_PAPER_RUN_MINIMUM_CLEAN_NEGATIVE_COUNT'] = paper_run_minimum_clean_negative_count
paper_run_dataset_minimum_count = os.environ.get('SLM_WM_PAPER_RUN_DATASET_QUALITY_MINIMUM_COUNT', '100')
os.environ['SLM_WM_PAPER_RUN_DATASET_QUALITY_MINIMUM_COUNT'] = paper_run_dataset_minimum_count
target_fpr_by_run = {'pilot_paper': '0.01', 'full_paper': '0.001'}
paper_run_target_fpr = os.environ.get('SLM_WM_PAPER_RUN_TARGET_FPR', target_fpr_by_run.get(paper_run_name, target_fpr_by_run['pilot_paper']))
os.environ['SLM_WM_PAPER_RUN_TARGET_FPR'] = paper_run_target_fpr
paper_run_target_fpr_token = paper_run_target_fpr.replace('.', '_')
os.environ['SLM_WM_PROTOCOL_PROFILE'] = f'{paper_run_name}_fixed_fpr_{paper_run_target_fpr_token}'
os.environ['SLM_WM_PROMPT_SET'] = paper_run_name
os.environ['SLM_WM_PROMPT_FILE'] = prompt_file_by_run.get(paper_run_name, prompt_file_by_run['pilot_paper'])
os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_DRIVE_OUTPUT_DIR'] = f'{drive_result_root}/external_baseline_official_reference'
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_OFFICIAL_OUTPUT_DIR', 'outputs/gaussian_shading_official_reference')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_OFFICIAL_SOURCE_DIR', 'external_baseline/primary/gaussian_shading/source')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_OFFICIAL_RUN_NAME', 'gaussian_shading_official_legacy_reference')
os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_SAMPLE_COUNT'] = paper_run_sample_count
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_OFFICIAL_MODEL_ID', 'Manojb/stable-diffusion-2-1-base')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_UPSTREAM_OFFICIAL_MODEL_ID', 'stabilityai/stable-diffusion-2-1-base')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_PATCH_MODEL_REPOSITORY_LAYOUT', '1')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_PREPARE_LOCAL_MODEL_REPOSITORY', '1')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_LOCAL_MODEL_REPOSITORY_DIR', '/content/gaussian_shading_model_repository/stable_diffusion_2_1_base')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_PATCH_MODEL_INDEX_FOR_LEGACY_TRANSFORMERS', '1')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_OFFICIAL_PYTHON_EXECUTABLE', '')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_PREPARE_LEGACY_ENV', '1')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_LEGACY_ENV_PREFIX', '/content/gaussian_shading_legacy_env')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_MICROMAMBA_PATH', '/content/bin/micromamba')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_LEGACY_PYTHON_VERSION', '3.8')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_LEGACY_TORCH_SPECS', 'torch==1.13.0+cu117 torchvision==0.14.0+cu117')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_STRICT_OFFICIAL_ENV', '1')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_ALLOW_COMPATIBLE_ENV_FALLBACK', '1')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_LEGACY_PYTORCH_INDEX_URL', 'https://download.pytorch.org/whl/cu117')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_LEGACY_PACKAGE_SPECS', 'transformers==4.23.1 diffusers==0.11.1 huggingface_hub==0.10.1 datasets==2.6.1 pyarrow<13 fsspec==2022.10.0 numpy==1.24.4 scipy==1.10.1 Pillow==9.5.0 tqdm==4.66.2 pycryptodome==3.20.0 open_clip_torch==2.7.0 ftfy==6.2.0 regex==2023.12.25 Requests==2.31.0 omegaconf==2.3.0 einops==0.4.1 kornia==0.6.4 matplotlib==3.7.5 timm==0.5.4')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_OFFICIAL_RUN_COMMAND', '1')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_OFFICIAL_REQUIRE_CUDA', '1')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_NUM_INFERENCE_STEPS', '50')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_NUM_INVERSION_STEPS', '50')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_USE_CHACHA', '1')
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_OFFICIAL_SUMMARY_IMPORT_PATH', '')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_OFFICIAL_LOG_IMPORT_PATH', '')
os.environ.setdefault('SLM_WM_GAUSSIAN_SHADING_OFFICIAL_TIMEOUT_SECONDS', '86400')
os.environ.setdefault('SLM_WM_ENABLE_WORKFLOW_PROGRESS_BAR', '1')
print({
    'drive_output_dir': os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_DRIVE_OUTPUT_DIR'],
    'enable_workflow_progress_bar': os.environ['SLM_WM_ENABLE_WORKFLOW_PROGRESS_BAR'],
    'sample_count': os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_SAMPLE_COUNT'],
    'official_model_id': os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_MODEL_ID'],
    'upstream_official_model_id': os.environ['SLM_WM_GAUSSIAN_SHADING_UPSTREAM_OFFICIAL_MODEL_ID'],
    'patch_model_repository_layout': os.environ['SLM_WM_GAUSSIAN_SHADING_PATCH_MODEL_REPOSITORY_LAYOUT'],
    'prepare_local_model_repository': os.environ['SLM_WM_GAUSSIAN_SHADING_PREPARE_LOCAL_MODEL_REPOSITORY'],
    'local_model_repository_dir': os.environ['SLM_WM_GAUSSIAN_SHADING_LOCAL_MODEL_REPOSITORY_DIR'],
    'patch_model_index_for_legacy_transformers': os.environ['SLM_WM_GAUSSIAN_SHADING_PATCH_MODEL_INDEX_FOR_LEGACY_TRANSFORMERS'],
    'prepare_legacy_env': os.environ['SLM_WM_GAUSSIAN_SHADING_PREPARE_LEGACY_ENV'],
    'legacy_env_prefix': os.environ['SLM_WM_GAUSSIAN_SHADING_LEGACY_ENV_PREFIX'],
    'legacy_torch_specs': os.environ['SLM_WM_GAUSSIAN_SHADING_LEGACY_TORCH_SPECS'],
    'strict_official_env': os.environ['SLM_WM_GAUSSIAN_SHADING_STRICT_OFFICIAL_ENV'],
    'compatible_env_fallback': os.environ['SLM_WM_GAUSSIAN_SHADING_ALLOW_COMPATIBLE_ENV_FALLBACK'],
    'official_python_executable': os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_PYTHON_EXECUTABLE'] or 'helper_managed_legacy_python',
    'run_command': os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_RUN_COMMAND'],
    'summary_import_path': os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_SUMMARY_IMPORT_PATH'],
    'log_import_path': os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_LOG_IMPORT_PATH'],
})


In [ ]:
%pip install -q --upgrade packaging huggingface_hub


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
from pathlib import Path

source_dir = Path(os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_SOURCE_DIR'])
requirements_path = source_dir / 'requirements.txt'
entrypoint_path = source_dir / 'run_gaussian_shading.py'
print({
    'source_dir': str(source_dir),
    'source_dir_exists_before_helper': source_dir.exists(),
    'entrypoint_exists_before_helper': entrypoint_path.exists(),
    'requirements_exists_before_helper': requirements_path.exists(),
    'source_prepare_policy': 'helper_will_clone_from_external_baseline_source_registry_when_missing',
})
if requirements_path.exists():
    print(requirements_path.read_text(encoding='utf-8')[:4000])


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)


In [ ]:
from pathlib import Path

from paper_workflow.colab_utils.gaussian_shading_official_reference import run_default_gaussian_shading_official_reference_plan

gaussian_shading_official_reference_summary = run_default_gaussian_shading_official_reference_plan(root='.')
legacy_prepare_path = Path('outputs/gaussian_shading_official_reference/gaussian_shading_legacy_environment_prepare_result.json')
model_prepare_path = Path('outputs/gaussian_shading_official_reference/gaussian_shading_model_repository_prepare_result.json')
for diagnostic_path in (legacy_prepare_path, model_prepare_path):
    if diagnostic_path.exists():
        print(diagnostic_path)
        print(diagnostic_path.read_text(encoding='utf-8')[:6000])
gaussian_shading_official_reference_summary


In [ ]:
import os
import subprocess
from datetime import datetime, timezone

from paper_workflow.colab_utils.gaussian_shading_official_reference import package_gaussian_shading_official_reference_outputs


def resolve_short_commit():
    try:
        result = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True, capture_output=True, text=True)
    except Exception:
        return 'git_unknown'
    return result.stdout.strip() or 'git_unknown'


drive_output_dir = os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_DRIVE_OUTPUT_DIR']
utc_suffix = datetime.now(timezone.utc).strftime('%Y%m%dt%H%M%sz')
short_commit = resolve_short_commit()
archive_name = f'external_baseline_official_reference_package_gaussian_shading_{utc_suffix}_{short_commit}.zip'
archive_record = package_gaussian_shading_official_reference_outputs(root='.', drive_output_dir=drive_output_dir, archive_name=archive_name)
archive_record.to_dict()


In [ ]:
from pathlib import Path

for result_path in sorted(Path('outputs/gaussian_shading_official_reference').rglob('*')):
    if result_path.is_file() and result_path.suffix.lower() in {'.json', '.jsonl', '.txt'}:
        print(result_path)
        print(result_path.read_text(encoding='utf-8', errors='ignore')[:4000])

expected_sample_count = paper_run_expected_sample_count
assert gaussian_shading_official_reference_summary['sample_count'] == expected_sample_count, gaussian_shading_official_reference_summary
if gaussian_shading_official_reference_summary['run_decision'] != 'pass':
    print({
        'gaussian_shading_official_reference_ready': gaussian_shading_official_reference_summary.get('gaussian_shading_official_reference_ready'),
        'unsupported_reason': gaussian_shading_official_reference_summary.get('unsupported_reason'),
        'legacy_environment_ready': gaussian_shading_official_reference_summary.get('legacy_environment_ready'),
        'official_command_return_code': gaussian_shading_official_reference_summary.get('official_command_return_code'),
    })
else:
    assert gaussian_shading_official_reference_summary['reference_import_ready'] is True, gaussian_shading_official_reference_summary
    assert gaussian_shading_official_reference_summary['governed_reference_record_count'] > 0, gaussian_shading_official_reference_summary
